# 第10集：NumPy复制

> 原视频 P10：2.8 numpy 的 copy & deep copy｜时长：6分46秒

本笔记严格依照原聊天记录中的讲解顺序整理。每个知识点先解释含义，再给出代码和记录中的预期输出；标为旧写法或故意报错的片段只用于阅读，不作为可执行单元。

这一集非常重要，核心问题是：

> `b = a` 到底有没有复制数组？

答案是：

```text
没有复制数组，只是多了一个指向同一数组的变量名。
```

原视频配套内容：[NumPy copy与deep copy](https://mofanpy.com/tutorials/data-manipulation/np-pd/np-copy)

## 1. `b = a`不是复制


In [ ]:
import numpy as np

a = np.arange(4)
b = a
c = a
d = b

print(a)
print(b)
print(c)
print(d)


输出：

```text
[0 1 2 3]
[0 1 2 3]
[0 1 2 3]
[0 1 2 3]
```

看起来有4个数组，实际上只有一个数组对象：

```text
a ──┐
b ──┤
c ──┼──→ [0 1 2 3]
d ──┘
```

---

## 2. 使用 `is` 判断是不是同一个对象


In [ ]:
print(b is a)
print(c is a)
print(d is a)


输出：

```text
True
True
True
```

`is` 判断的不是数值是否相等，而是：

> 两个变量是否指向内存中的同一个对象。

对比：


In [ ]:
x = np.array([1, 2, 3])
y = np.array([1, 2, 3])

print(np.array_equal(x, y))
print(x is y)


输出：

```text
True
False
```

- `array_equal`：内容相同
- `is`：是不是同一个对象

虽然 `x` 和 `y` 内容一样，但它们是分别创建的两个数组。

---

## 3. 通过a修改数组


In [ ]:
a[0] = 11

print(a)
print(b)
print(c)
print(d)


输出：

```text
[11  1  2  3]
[11  1  2  3]
[11  1  2  3]
[11  1  2  3]
```

因为这4个变量指向同一个数组。

不是 Python 同时修改了4份数据，而是：

```text
实际上只有一份数据被修改了。
```

---

## 4. 通过d修改，其他变量也受影响


In [ ]:
d[1:3] = [22, 33]

print(a)
print(b)
print(c)
print(d)


输出：

```text
[11 22 33  3]
[11 22 33  3]
[11 22 33  3]
[11 22 33  3]
```

切片：

```python
d[1:3]
```

对应索引1和2：

```text
[1, 2]
```

把它们修改为：

```text
[22, 33]
```

最终共同指向的数组变成：

```text
[11 22 33 3]
```

---

## 5. 使用 `copy()`创建独立数组


In [ ]:
a = np.array([11, 22, 33, 3])
b = a.copy()

print(a)
print(b)
print(b is a)


输出：

```text
[11 22 33  3]
[11 22 33  3]
False
```

此时内存关系是：

```text
a ──→ [11 22 33 3]

b ──→ [11 22 33 3]
```

看起来内容相同，但它们是独立的数据。

修改 `a`：


In [ ]:
a[3] = 44

print(a)
print(b)


输出：

```text
[11 22 33 44]
[11 22 33  3]
```

`b` 不受影响。

---

## 6. NumPy切片通常是视图

这一点非常容易踩坑：


In [ ]:
a = np.array([10, 20, 30, 40])
b = a[1:3]

print(b)


输出：

```text
[20 30]
```

现在修改 `b`：


In [ ]:
b[0] = 999

print(b)
print(a)


输出：

```text
[999  30]
[ 10 999  30  40]
```

为什么修改切片 `b`，原数组 `a` 也改变了？

因为 NumPy 的普通切片通常返回：

```text
视图
```

视图不是独立数据，而是从另一个角度查看原数组的一部分：

```text
a ──→ [10, 20, 30, 40]
           ↑   ↑
b ─────────┘   └── 查看a的这一部分
```

---

## 7. 如果希望切片完全独立

需要主动调用：

```python
copy()
```


In [ ]:
a = np.array([10, 20, 30, 40])
b = a[1:3].copy()

b[0] = 999

print(a)
print(b)


输出：

```text
[10 20 30 40]
[999  30]
```

这一次 `a` 没有变化。

---

## 8. 检查两个数组是否共享内存


In [ ]:
a = np.array([10, 20, 30, 40])

b = a
c = a[1:3]
d = a.copy()

print(np.shares_memory(a, b))
print(np.shares_memory(a, c))
print(np.shares_memory(a, d))


输出：

```text
True
True
False
```

解释：

```text
b = a       → 完全是同一个数组
c = a[1:3]  → 通常是共享数据的视图
d = a.copy() → 独立数组
```

---

## 9. “浅复制”和“深复制”怎么理解

原视频把：


In [ ]:
b = a


称为有关联的赋值，把：


In [ ]:
b = a.copy()


称为 deep copy。

初学阶段可以这样记：

```text
b = a      → 不复制，共用同一份数组
b = a.copy() → 复制数据，两个数组相互独立
```

但严格来说，对普通数字数组，`a.copy()` 会复制数组的数据缓冲区，已经足以实现独立修改。

如果数组中保存的是复杂 Python 对象，`copy()` 对对象内部更深层的数据不一定全部递归复制。这是以后接触 `dtype=object` 时才需要关心的问题。

---

# 第6～10集总复习

| 知识 | 作用 |
|---|---|
| `np.argmin(A)` | 最小值的展开索引 |
| `np.argmax(A)` | 最大值的展开索引 |
| `np.mean(A)` | 平均值 |
| `np.average(A)` | 平均值，可指定权重 |
| `np.median(A)` | 中位数 |
| `np.cumsum(A)` | 累加 |
| `np.diff(A)` | 相邻元素之差 |
| `np.nonzero(A)` | 非零元素坐标 |
| `np.sort(A)` | 返回排序后的新数组 |
| `A.T` | 数组转置 |
| `np.clip(A, 5, 9)` | 把数值限制在5～9 |
| `A[行, 列]` | 二维索引 |
| `A[:, 1]` | 取出第1列 |
| `A[1, :]` | 取出第1行 |
| `A.flatten()` | 展开成一维数组 |
| `np.vstack()` | 上下合并 |
| `np.hstack()` | 左右合并 |
| `np.newaxis` | 增加维度 |
| `np.concatenate()` | 按指定轴合并 |
| `np.split()` | 等量分割 |
| `np.array_split()` | 允许不等量分割 |
| `np.vsplit()` | 拆分行 |
| `np.hsplit()` | 拆分列 |
| `b = a` | 两个变量共用同一数组 |
| `b = a.copy()` | 创建独立数组 |

这五集最需要牢牢记住的是：

```text
1. axis=0通常沿行方向压缩或操作，得到每列结果。
2. axis=1通常沿列方向压缩或操作，得到每行结果。
3. 一维数组的.T不会把它变成列数组，要用[:, np.newaxis]。
4. split要求等分，array_split允许不等分。
5. b=a没有复制，b=a.copy()才创建独立数据。
```
